# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and accessed via the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print the dataset title and description
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The `mlcroissant` library provides access to record sets via their `@id`. To list all available record sets and their details:

In [ ]:
# List all record sets in the dataset by their @id
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', 'No description')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) [Type: {field.data_type}] [Column: {field.column.id if hasattr(field, 'column') else 'N/A'}]")
    print()
# For demonstration, show records from the first available record set
if len(record_sets) > 0:
    first_record_set_id = record_sets[0].id
    print(f"\nSample records from RecordSet (@id={first_record_set_id}):")
    for ix, rec in enumerate(dataset.records(record_set=first_record_set_id)):
        print(rec)
        if ix >= 2:
            break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

Each record set is referenced by its `@id`. We'll load all available record sets and preview their contents.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Record Sets IDs: {record_set_ids}\n")

# Load each record set as a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for {record_set_id}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric mental health score field, filter, normalize, and group by a demographic attribute. Reference fields by their `@id` throughout.

In [ ]:
# For demonstration, select the main survey record set (e.g., first one) and numeric fields
if len(dataframes):
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]

    # Identify numeric fields by inspecting types or using prior knowledge
    numeric_fields = [col for col in df.columns if ('score' in col or col.startswith('GAD') or col.startswith('PHQ') or col.startswith('PSQ') or df[col].dtype in ['float64','int64'])]
    print(f"Numeric fields: {numeric_fields}")

    # Choose one: e.g., '@id: phq9_score' if present
    numeric_field_id = None
    for col in numeric_fields:
        if 'phq' in col.lower():
            numeric_field_id = col
            break
    if not numeric_field_id and len(numeric_fields):
        numeric_field_id = numeric_fields[0]

    threshold = 10
    print(f"\nFiltering records with {numeric_field_id} > {threshold}...")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic field, e.g., '@id: gender' or 'age_group'
    group_fields = [col for col in df.columns if ('gender' in col.lower() or 'age' in col.lower() or 'education' in col.lower())]
    group_field_id = group_fields[0] if len(group_fields) else None

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field and also visualize the grouping from section 4.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if len(dataframes):
    df = dataframes[main_record_set_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
        plt.title(f"Distribution of {numeric_field_id} (@id)")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

    # Plot mean by group_field if available
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8, 5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (@id)")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. We loaded data via Croissant schema, explored available record sets and fields by their `@id`, filtered and normalized mental health scores, and visualized results. These steps show how `mlcroissant` enables reproducible and standards-based analysis of FAIR datasets.

Further steps can include advanced statistical modeling or integration with other datasets.